In [ ]:
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt
import re

ROOT_DIR = Path.cwd().parent.parent.parent

lkp_folder = os.path.join(ROOT_DIR, 'external_data','lookups','EHR','primary_care')

from pomegranate.db.ukbdb import UKBDatabase
Database = UKBDatabase()

## Expand BNF codes based on descriptions

Expand the BNF codes based on mapping broad codes to more specific codes via drug descriptions.

TWO CASES:
- CASE A: Internal BNF expansion
    - entry_desc matches lkp_desc for a lkp_code that is an expansion of the entry_code
- CASE B: BNF expansion via BNF:dm+d maps
    - entry_desc matches a dmd_lkp_desc to a dmd_code which is mapped to a bnf_lkp_code that is an expansion of the entry_code


Will require a map file that maps both by **entry code** and by **entry description**.

### Pull codes and descriptions from original entries

In [ ]:
bnf_desc_df = pd.DataFrame(Database.query(
    f'''
    SELECT bnf_code, drug_name, COUNT(*)
    FROM gp_prescriptions 
    WHERE 
        bnf_code IS NOT NULL AND 
        dmd_code IS NULL AND
        drug_name IS NOT NULL
    GROUP BY bnf_code, drug_name;
    '''
    ).fetchall(), columns=['entry_bnf','desc','n_entries'])

Reformat BNF codes, convert descriptions to lowercase, and sum per unique code:entry

In [ ]:
# standardise code format (to match SNOMED and BNF lookup)
def format_bnf(code):
    # drop trailing 0s
    formatted_code = code.replace('.00','').replace('.','')
    # if still has trailing zeroes
    if formatted_code[-2:]=='00':
        return formatted_code[:-2]
    else:
        return formatted_code

In [ ]:
# lowercase
bnf_desc_df['lower_desc'] = bnf_desc_df['desc'].apply(str.lower)
# reformat
bnf_desc_df['formatted_entry_bnf'] = bnf_desc_df['entry_bnf'].apply(format_bnf)

In [ ]:
# group by unique combinations
bnf_desc_df_reformatted = (bnf_desc_df
                           .copy()
                           .drop(columns=['desc','entry_bnf'])
                           .groupby(['formatted_entry_bnf','lower_desc'], as_index=False)
                           .sum())

In [ ]:
# summarize
print(f'''There are {bnf_desc_df_reformatted["formatted_entry_bnf"].nunique()} reformatted BNF codes with descriptions
There are {len(bnf_desc_df_reformatted)} unique combinations
They account for {bnf_desc_df_reformatted["n_entries"].sum()} entries''')

### Edge case - '99' BNF codes

Not enough to worry about

In [ ]:
bnf_desc_df_reformatted[bnf_desc_df_reformatted['formatted_entry_bnf']=='99']['n_entries'].sum()

### Does the description match one in the BNF lookup?

In [ ]:
bnf_lkp = pd.read_csv(os.path.join(lkp_folder,'bnf_key.csv.gz'))
# lowercase
bnf_lkp['lower_desc'] = bnf_lkp['desc'].apply(str.lower)

In [ ]:
bnf_desc_df_reformatted = (bnf_desc_df_reformatted
 .merge(bnf_lkp, how='left', on='lower_desc')
 .rename(columns={'bnf_code':'lkp_bnf', 'lower_desc':'entry_desc', 'desc':'bnf_lkp_desc'})
 )

#### No

In [ ]:
bnf_no_bnf_lkp_match = bnf_desc_df_reformatted[bnf_desc_df_reformatted['lkp_bnf'].isnull()].copy()

In [ ]:
print(f'{len(bnf_no_bnf_lkp_match)} entry combinations do not have a match in the BNF lookup\nThey account for {bnf_no_bnf_lkp_match["n_entries"].sum()} entries')

#### Yes

In [ ]:
bnf_bnf_lkp_match = bnf_desc_df_reformatted.dropna().copy()

In [ ]:
print(f'{len(bnf_bnf_lkp_match)} entry combinations have a match in the BNF lookup\nThey account for {bnf_bnf_lkp_match["n_entries"].sum()} entries')

#### Match to entry code

In [ ]:
bnf_bnf_lkp_match['entry_match'] = bnf_bnf_lkp_match['formatted_entry_bnf']==bnf_bnf_lkp_match['lkp_bnf']

In [ ]:
bnf_bnf_entry_match = bnf_bnf_lkp_match[bnf_bnf_lkp_match['entry_match']==True].copy()

In [ ]:
print(f'{len(bnf_bnf_entry_match)} combinations match desc and entry code to the lookup\nThey account for {bnf_bnf_entry_match["n_entries"].sum()} entries')

#### Match to expanded code

Add limitation for 6 characters

In [ ]:
# filter for those matching
def check_bnf_start(row):
    entry_code = row['formatted_entry_bnf']
    lkp_code = row['lkp_bnf']
    if re.match(f'^{entry_code}',lkp_code):
        return True
    # sometimes the last value is irrelevant as its replaced by a letter, so secondary check
    # ONLY DO IF THE CODE IS AT LEAST 6 CHARACTERS LONG - to prevent replacing something like '19' to '1'
    elif len(entry_code)>=6:
        if re.match(f'^{entry_code[:-1]}',lkp_code):
            return True
        else:
            return False
    else:
        return False

In [ ]:
bnf_bnf_lkp_match['expanded_match'] = bnf_bnf_lkp_match.apply(lambda row: check_bnf_start(row), axis=1)

In [ ]:
bnf_expanded_lkp_match = bnf_bnf_lkp_match[
    (bnf_bnf_lkp_match['expanded_match']==True) &
    (bnf_bnf_lkp_match['entry_match']==False)].copy()

In [ ]:
print(f'{len(bnf_expanded_lkp_match)} combinations match a desc to an expanded entry code\nThey account for {bnf_expanded_lkp_match["n_entries"].sum()} entries')

#### Match to different code

In [ ]:
bnf_diff_match = bnf_bnf_lkp_match[
    (bnf_bnf_lkp_match['expanded_match']==False) &
    (bnf_bnf_lkp_match['entry_match']==False)].copy()

In [ ]:
print(f'{len(bnf_diff_match)} combinations match a desc to a different BNF code than in the entry\nThey account for {bnf_diff_match["n_entries"].sum()} entries')

### Does the description match to the dm+d lookup?

In [ ]:
dmd_key = pd.read_csv(os.path.join(lkp_folder,'dmd_desc_type_lkp.csv.gz'), compression='gzip', dtype=object)
# lowercase
dmd_key['lower_desc'] = dmd_key['desc'].apply(str.lower)

In [ ]:
bnf_dmd_desc_match = (bnf_no_bnf_lkp_match
 .drop(columns=['bnf_lkp_desc','lkp_bnf','bnf_type'])
 .merge(dmd_key[['reference_code','dmd_type','lower_desc']], 
        how='left', 
        left_on='entry_desc',
          right_on='lower_desc')
)

#### No

In [ ]:
bnf_no_desc_match = bnf_dmd_desc_match[bnf_dmd_desc_match['reference_code'].isnull()].copy()

In [ ]:
print(f'{len(bnf_no_desc_match)} combinations have no matches at all by description\nThey account for {bnf_no_desc_match["n_entries"].sum()} entries')

#### Yes

In [ ]:
bnf_dmd_match = bnf_dmd_desc_match.dropna().copy()

In [ ]:
print(f'{len(bnf_dmd_match)} combinations match by dm+d description\nThey account for {bnf_dmd_match["n_entries"].sum()} entries')

### Map dmd:BNF?

In [ ]:
bnf_dmd_map = pd.read_csv(os.path.join(lkp_folder,'bnf_dmd_map_simple.csv'), dtype='object')

In [ ]:
bnf_dmd_match = (bnf_dmd_match
# map to dmd
 .merge(bnf_dmd_map, how='left', left_on='reference_code', right_on='reference_dmd_code')
 .drop(columns=['reference_dmd_code','lower_desc'])
 .rename(columns={'bnf_code':'lkp_bnf'})
# add BNF info
 .merge(bnf_lkp[['bnf_code','bnf_type','lower_desc']], how='left', left_on='lkp_bnf', right_on='bnf_code')
 .drop(columns=['bnf_code'])
 .rename(columns={'lower_desc':'lkp_bnf_desc','bnf_type':'lkp_bnf_type'}))

#### No

In [ ]:
bnf_dmd_no_bnf_match = bnf_dmd_match[bnf_dmd_match['lkp_bnf'].isnull()].copy()

In [ ]:
print(f'{len(bnf_dmd_no_bnf_match)} combinations matched by dm+d description dont map to BNF\nThey account for {bnf_dmd_no_bnf_match["n_entries"].sum()} entries')

#### Yes

In [ ]:
bnf_dmd_bnf_match = bnf_dmd_match.dropna().copy()

In [ ]:
print(f'{len(bnf_dmd_bnf_match)} combinations matched by dm+d description map to BNF\nThey account for {bnf_dmd_bnf_match["n_entries"].sum()} entries')

### Is the mapped BNF an extension of the entry code?

In [ ]:
bnf_dmd_bnf_match['expanded_match'] = bnf_dmd_bnf_match.apply(lambda row: check_bnf_start(row), axis=1)

In [ ]:
bnf_dmd_bnf_match

#### No

In [ ]:
bnf_dmd_bnf_no_match = bnf_dmd_bnf_match[bnf_dmd_bnf_match['expanded_match']==False].copy()

In [ ]:
print(f'{len(bnf_dmd_bnf_no_match)} combinations matched by dm+d description map to a BNF that is NOT an extension of the entry\nThey account for {bnf_dmd_bnf_no_match["n_entries"].sum()} entries')

In [ ]:
bnf_dmd_bnf_no_match.head()

#### Yes

In [ ]:
bnf_dmd_bnf_expansion = bnf_dmd_bnf_match[bnf_dmd_bnf_match['expanded_match']==True].copy()

In [ ]:
print(f'{len(bnf_dmd_bnf_expansion)} combinations matched by dm+d description map to a BNF that IS an extension of the entry\nThey account for {bnf_dmd_bnf_expansion["n_entries"].sum()} entries')

In [ ]:
bnf_dmd_bnf_expansion.head()

### Put expanded codes in table to map in DB

Link the expansion back to their original codes/descs

In [ ]:
bnf_expanded_lkp_match_toadd = (bnf_desc_df
    # combine with the original code and entry description
    .merge(bnf_expanded_lkp_match
            [['formatted_entry_bnf','entry_desc','lkp_bnf']], 
             how='right', 
             left_on=['formatted_entry_bnf','lower_desc'], 
             right_on=['formatted_entry_bnf','entry_desc'])
    # drop unneeded columns
    .drop(columns=['n_entries','lower_desc','entry_desc','formatted_entry_bnf']))

In [ ]:
bnf_dmd_bnf_expansion_toadd = (bnf_desc_df
    # combine with the original code and entry description
    .merge(bnf_dmd_bnf_expansion
            [['formatted_entry_bnf','entry_desc','lkp_bnf']], 
             how='right', 
             left_on=['formatted_entry_bnf','lower_desc'], 
             right_on=['formatted_entry_bnf','entry_desc'])
    # drop unneeded columns
    .drop(columns=['n_entries','lower_desc','entry_desc','formatted_entry_bnf']))

In [ ]:
expanded_match_df = pd.concat([bnf_dmd_bnf_expansion_toadd, bnf_expanded_lkp_match_toadd]).drop_duplicates(ignore_index=True)

One odd case where multiple expanded BNF

In [ ]:
expanded_match_df.groupby(['entry_bnf','desc'])['lkp_bnf'].nunique().reset_index().sort_values('lkp_bnf', ascending=False).head()

In [ ]:
expanded_match_df[expanded_match_df['entry_bnf']=='15.01.02.02.00']

Choosing 1501020R0 because its what OpenPrescribing uses https://openprescribing.net/chemical/1501020R0/

In [ ]:
# drop the other one
expanded_match_df = expanded_match_df[~(expanded_match_df['entry_bnf']=='15.01.02.02.00')&~(expanded_match_df['lkp_bnf']=='1501020R0BB')]

In [ ]:
expanded_match_df.groupby(['entry_bnf','desc'])['lkp_bnf'].nunique().reset_index().sort_values('lkp_bnf', ascending=False).head()

In [ ]:
# export
expanded_match_df.to_csv(os.path.join(lkp_folder,'bnf_code_expansion_by_desc_map.csv'), index=False)

## BNF to dm+d

Want to make the map file to map to the version of dm+d codes used in the entries. Preferably to VMP codes, as that is the most common (and useful) form to map to

In [ ]:
# BNF-SNOMED map
bnf_snomed_path = os.path.join(lkp_folder,'BNF Snomed Mapping data 20230622.csv.gz')
bnf_snomed_map = pd.read_csv(bnf_snomed_path, dtype=object, compression='gzip')

# map the dmd codes to the reference
dmd_ref_map = dmd_key[['dmd_code','reference_code']].drop_duplicates()
dmd_ref_map = dict(zip(dmd_ref_map['dmd_code'], dmd_ref_map['reference_code']))

bnf_snomed_map.insert(6, 'reference_dmd_code', bnf_snomed_map['SNOMED Code'].map(dmd_ref_map))

In [ ]:
# get the codes mapped to the lowest level of dm+d possible (e.g., VMP then AMP, then VMPP)
dmd_order = ['VMP','AMP','VMPP','AMPP']
bnf_snomed_map['dmd_type'] = pd.Categorical(bnf_snomed_map['VMP / VMPP/ AMP / AMPP'], categories=dmd_order, ordered=True)

bnf_dmd_map = (bnf_snomed_map
                .sort_values(by='dmd_type')
                .drop_duplicates(subset='BNF Code', keep='first') # just keep the first (lowest) mapped dm+d code
                .rename(columns={'BNF Code':'bnf_code'})
                [['bnf_code','reference_dmd_code']])

assert bnf_dmd_map['bnf_code'].nunique() == len(bnf_dmd_map)

In [ ]:
print(len(bnf_dmd_map[bnf_dmd_map['reference_dmd_code'].isna()]), 'lookup BNF codes werent able to be mapped to a reference dm+d code')

In [ ]:
# check which in the lkp
bnf_key = pd.read_csv(os.path.join(lkp_folder,'bnf_key.csv.gz'),compression='gzip',dtype=object)

print('BNF codes in the map file that arent in the lookup:')
set(bnf_dmd_map['bnf_code']) - set(bnf_key['bnf_code'])

Pretty good coverage, can't really improve much more

In [ ]:
# check the coverage for the entries that need to be mapped
bnf_in_entries = pd.DataFrame(Database.query(
    '''
    SELECT DISTINCT(clean_entry_code) 
    FROM gp_prescriptions_harmonized
    WHERE entry_ontology = 'bnf';
    '''
).fetchall())
bnf_in_entries = list(bnf_in_entries[0])
# get full info for these codes
bnf_in_entries = bnf_key[bnf_key['bnf_code'].isin(bnf_in_entries)]

In [ ]:
bnf_key[bnf_key['bnf_code'].isin(bnf_dmd_map['bnf_code'])]['bnf_type'].value_counts()

In [ ]:
# the relevant ones are either presentation or product codes, so filter the entry codes by this
bnf_in_entries = bnf_in_entries[bnf_in_entries['bnf_type'].isin(['6_product','7_presentation'])]
entry_bnf_dmd_map = bnf_dmd_map[bnf_dmd_map['bnf_code'].isin(bnf_in_entries['bnf_code'])]

In [ ]:
print('Out of the',bnf_in_entries['bnf_code'].nunique(),'BNF codes in the entries,',entry_bnf_dmd_map['bnf_code'].nunique(), 'are represented in the map file')

Can I map the missing ones myself via description matching?

In [ ]:
bnf_missing_map = list(set(bnf_in_entries['bnf_code']) - set(entry_bnf_dmd_map['bnf_code']))
bnf_missing_map = bnf_key[bnf_key['bnf_code'].isin(bnf_missing_map)].drop(columns=['bnf_type'])

In [ ]:
# merge, prioritizing VMP and choosing first code
bnf_mapped_by_desc = (bnf_missing_map
                    .merge(dmd_key, how='inner', on='desc')
                    .sort_values('dmd_type', ascending=False)
                    .drop_duplicates(subset='bnf_code', keep='first')
                    [['bnf_code','reference_code']]
                    .rename(columns={'reference_code':'reference_dmd_code'}))

In [ ]:
print('Of the',bnf_missing_map['bnf_code'].nunique(),'BNF codes missing a map, I was able to map',bnf_mapped_by_desc['bnf_code'].nunique(),'by description matching')

In [ ]:
# add to map file
bnf_dmd_map = pd.concat([bnf_mapped_by_desc,bnf_dmd_map], ignore_index=True)

Filter out BNF codes that aren't in the lookup and any NA dm+d

In [ ]:
bnf_dmd_map = bnf_dmd_map[
    (~bnf_dmd_map['reference_dmd_code'].isna()) &
    (bnf_dmd_map['bnf_code'].isin(bnf_key['bnf_code']))
    ].copy()

Export final simple map file

In [ ]:
# export
bnf_dmd_map.to_csv(os.path.join(lkp_folder,'bnf_dmd_map_simple.csv'), index=False)

## Welsh Read2 to dm+d 

In [ ]:
welsh_lkp_dmd = pd.read_csv(os.path.join(lkp_folder,'welsh_read_from_entries_lkp_dmd_map_via_UMLS.csv'), dtype=object)
dmd_key = pd.read_csv(os.path.join(lkp_folder,'dmd_desc_type_lkp.csv.gz'), compression='gzip', dtype=object)

How many of the codes mapped via UMLS are in the lookup?

In [ ]:
print(f'''Of the {welsh_lkp_dmd['welsh_read_code'].nunique()} Welsh Read2 codes in the entries, \
{welsh_lkp_dmd.dropna(subset=['umls_dmd_code'])['welsh_read_code'].nunique()} codes were linked to {welsh_lkp_dmd['umls_dmd_code'].nunique()} dm+d codes via UMLS.
{len(set(welsh_lkp_dmd['umls_dmd_code']).intersection(set(dmd_key['dmd_code'])))} of those dm+d codes are in the dm+d lookup I built.''')

In [ ]:
welsh_dmd_map_from_lkp = (welsh_lkp_dmd
                        .merge(dmd_key[['dmd_code','reference_code','dmd_type']].drop_duplicates(), 
                        how='inner', left_on='umls_dmd_code', right_on='dmd_code')
                        [['welsh_read_code','reference_code','dmd_type']]
                        .rename(columns={'reference_code':'dmd_reference_code'})
                        .sort_values('dmd_type')
                        .drop_duplicates(subset='welsh_read_code', keep='first')
                        .drop(columns=['dmd_type'])) # like before, just use one reference code

assert welsh_dmd_map_from_lkp['welsh_read_code'].nunique() == len(welsh_dmd_map_from_lkp)
print(welsh_dmd_map_from_lkp['welsh_read_code'].nunique(), 'Read codes were mapped to', welsh_dmd_map_from_lkp['dmd_reference_code'].nunique(),'reference dm+d codes via the UMLS')

Remove those, then try merging to description

In [ ]:
welsh_to_map = welsh_lkp_dmd[~welsh_lkp_dmd['welsh_read_code'].isin(welsh_dmd_map_from_lkp['welsh_read_code'])]

In [ ]:
# combine the two available descriptions
welsh_lkp_desc_umls_read = welsh_to_map[['welsh_read_code','umls_dmd_code','umls_read_desc']].dropna(subset='umls_read_desc').rename(columns={'umls_read_desc':'desc'})
welsh_lkp_desc_umls_read['source'] = 'umls_read'

welsh_lkp_desc_umls_dmd = welsh_to_map[['welsh_read_code','umls_dmd_code','umls_dmd_desc']].dropna(subset='umls_dmd_desc').rename(columns={'umls_dmd_desc':'desc'})
welsh_lkp_desc_umls_dmd['source'] = 'umls_dmd'

#combine
welsh_to_map = pd.concat([welsh_lkp_desc_umls_read,welsh_lkp_desc_umls_dmd], ignore_index=True)
# convert to lowercase
welsh_to_map['desc'] = welsh_to_map['desc'].apply(str.lower)
dmd_key['desc'] = dmd_key['desc'].apply(str.lower)

In [ ]:
# merge
welsh_dmd_map_via_desc = (welsh_to_map
                          .merge(dmd_key[['desc','reference_code']].drop_duplicates(), how='inner', on='desc')
                          [['welsh_read_code','reference_code']]
                         .rename(columns={'reference_code':'dmd_reference_code'})
                         .drop_duplicates(subset='welsh_read_code', keep='first')) # like before, just use one reference code

assert welsh_dmd_map_via_desc['welsh_read_code'].nunique() == len(welsh_dmd_map_via_desc)
print(welsh_dmd_map_via_desc['welsh_read_code'].nunique(), 'Read codes were mapped to', welsh_dmd_map_via_desc['dmd_reference_code'].nunique(),'reference dm+d codes via descriptions provided by UMLS')

Add to map file

Remove these and try using the entries

In [ ]:
welsh_to_map = welsh_to_map[~welsh_to_map['welsh_read_code'].isin(welsh_dmd_map_via_desc['welsh_read_code'])].copy()

I'm making the assumption that Read descriptions are consistent (okay in my checks and I think also integrated in the UMLS mapping) among providers, where they're the same

In [ ]:
read_codes_str = ','.join(f"'{code}'" for code in welsh_to_map['welsh_read_code'])
                         
welsh_read_entry_desc = pd.DataFrame(Database.query(
    f'''
    SELECT read_2, drug_name, COUNT(*)
    FROM gp_prescriptions 
    WHERE 
        read_2 IN ({read_codes_str}) AND 
        drug_name IS NOT NULL
    GROUP BY read_2, drug_name;
    '''
    ).fetchall(), columns=['read_code','desc','n_entries'])

In [ ]:
welsh_read_entry_desc['read_code'].nunique()

Remove the brands from the descriptions

In [ ]:
def remove_brand_suffix(s):
    return re.sub(r'\s*\(([^)]*|\.\.\.)\)?$', '', s)

In [ ]:
welsh_read_entry_desc['formatted_desc'] = welsh_read_entry_desc['desc'].apply(remove_brand_suffix).apply(str.lower)

In [ ]:
# sum by description
welsh_read_entry_desc = (welsh_read_entry_desc
 .drop(columns='desc')
 .groupby(['read_code','formatted_desc'], as_index=False)
 .sum())

Sort for those with only a single description

In [ ]:
welsh_read_entry_desc_counts = pd.DataFrame(welsh_read_entry_desc['read_code'].value_counts()).reset_index()
welsh_read_entry_desc_counts.columns = ['read_code','n_desc']

In [ ]:
welsh_read_entry_one_desc = welsh_read_entry_desc[
    welsh_read_entry_desc['read_code'].isin(
        welsh_read_entry_desc_counts[welsh_read_entry_desc_counts['n_desc']==1]['read_code'].values
    )
]

Merge with dm+d by description

In [ ]:
welsh_dmd_map_via_entry_desc = (welsh_read_entry_one_desc
                            .merge(dmd_key[['desc','reference_code','dmd_type']]
                                    .drop_duplicates(), 
                                    how='inner', 
                                    left_on='formatted_desc',
                                    right_on='desc')
                            .drop(columns=['formatted_desc','desc','n_entries'])
                            .sort_values('dmd_type', ascending=False)
                            .drop_duplicates(subset='read_code', keep='first')
                            .drop(columns=['dmd_type'])
                            .rename(columns={'reference_code':'dmd_reference_code','read_code':'welsh_read_code'}))

Combine and export

In [ ]:
welsh_dmd_map = (pd.concat([welsh_dmd_map_from_lkp, 
                            welsh_dmd_map_via_desc, 
                            welsh_dmd_map_via_entry_desc])
                    .drop_duplicates(subset='welsh_read_code', ignore_index=True))
print('In total',welsh_dmd_map['welsh_read_code'].nunique(), 'Read codes were mapped to', welsh_dmd_map['dmd_reference_code'].nunique(),'reference dm+d codes')

In [ ]:
# export
welsh_dmd_map.to_csv(os.path.join(lkp_folder,'welsh_read_dmd_map_simple.csv'), index=False)